# 09 — Test-set inference for every saved run

For each `runs/<run_id>/result.json`, re-evaluate the same config on the **test** set
(originally the ImageNet validation split at `/home/pf4636/imagenet2/val`) and write a
fresh result JSON to `runs/final_runs/<run_id>/`.

- pytorch backend → load weights → eval on test loader
- torchao_cpu_ptq → re-calibrate on train loader → eval on test loader
- tensorrt → reuse existing engine in `engines/` → trt_evaluate on test loader

In [ ]:
import sys, json, time
from pathlib import Path
from dataclasses import replace

SRC = Path("..").resolve() / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from config import ExperimentConfig, set_seed
from data import build_imagenet_dataset, build_runner_loaders
from model import get_model
from precision import apply_precision
from quant_ptq_cpu import quantize_int8_x86_pt2e
from eval import evaluate
from utils import iter_result_jsons, read_json, write_json
from runner import _get_trt_paths

TEST_ROOT  = "/home/pf4636/imagenet2"
RUNS_IN    = Path("../runs/val_infer").resolve()
RUNS_OUT   = Path("../runs/final_runs").resolve()
RUNS_OUT.mkdir(parents=True, exist_ok=True)
print(f"reading from: {RUNS_IN}")
print(f"writing to  : {RUNS_OUT}")

In [ ]:
def build_test_loader(cfg: ExperimentConfig) -> DataLoader:
    test_cfg = replace(cfg, imagenet_path=TEST_ROOT)
    dataset = build_imagenet_dataset(test_cfg, split="val")
    pin_memory = str(cfg.device).startswith("cuda")
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
        drop_last=True,
    )

def run_on_test(cfg: ExperimentConfig, criterion=None):
    cfg = cfg.normalized()
    cfg.validate()
    set_seed(cfg)
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    test_loader = build_test_loader(cfg)
    t0 = time.perf_counter()

    if cfg.backend == "pytorch":
        model = apply_precision(get_model(cfg), cfg)
        tracker = evaluate(model, test_loader, cfg, criterion=criterion)

    elif cfg.backend == "torchao_cpu_ptq":
        train_loader, _ = build_runner_loaders(cfg)
        model = quantize_int8_x86_pt2e(
            get_model(cfg), train_loader,
            calib_num_batches=cfg.cpu_calib_num_batches,
        )
        tracker = evaluate(model, test_loader, cfg, criterion=criterion)

    elif cfg.backend == "tensorrt":
        from trt_infer import trt_evaluate
        from onnx_exporter import ONNXExporter
        from trt_builder import build_engine

        onnx_path, engine_path, _ = _get_trt_paths(cfg)
        if not onnx_path.exists():
            ONNXExporter(get_model(cfg), cfg.device, onnx_path).export_model(
                opset_version=cfg.trt_opset if cfg.trt_opset > 1 else 17,
                dynamic_batch=True,
                dummy_input_shape=(1, 3, 224, 224),
            )
        if not engine_path.exists():
            build_engine(
                onnx_path=onnx_path,
                engine_path=engine_path,
                precision=cfg.model_precision,
                batch_size=cfg.batch_size,
                workspace_mb=cfg.trt_workspace_mb,
            )
        tracker = trt_evaluate(engine_path, cfg, test_loader, criterion)

    else:
        raise ValueError(f"Unknown backend: {cfg.backend}")

    payload = {
        "status": "ok",
        "run_id": cfg.run_id(),
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "results": tracker.summary(),
        "error": None,
        "total_eval_time_sec": round(time.perf_counter() - t0, 3),
    }
    return payload, tracker

In [ ]:
def cfg_from_payload(payload: dict) -> ExperimentConfig:
    raw = payload["config"]
    fields = {f for f in ExperimentConfig.__dataclass_fields__}
    return ExperimentConfig(**{k: v for k, v in raw.items() if k in fields})

configs = []
for p in iter_result_jsons(RUNS_IN):
    if RUNS_OUT in p.parents:
        continue
    payload = read_json(p)
    if payload.get("status") != "ok":
        continue
    configs.append(cfg_from_payload(payload))

print(f"found {len(configs)} runs to re-evaluate on test set")
for c in configs:
    print("  ", c.run_id())

In [ ]:
SKIP_EXISTING = True

results_summary = []
for i, cfg in enumerate(configs, 1):
    out_path = RUNS_OUT / cfg.run_id() / "result.json"
    print(f"\n[{i}/{len(configs)}] {cfg.run_id()}")

    if SKIP_EXISTING and out_path.exists():
        print(f"  already exists, skipping: {out_path}")
        results_summary.append(read_json(out_path))
        continue

    try:
        payload, _ = run_on_test(cfg)
        write_json(out_path, payload)
        print(f"  saved: {out_path}")
        r = payload["results"]
        print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.3f} ms")
        results_summary.append(payload)
    except Exception as e:
        err_payload = {
            "status": "error",
            "run_id": cfg.run_id(),
            "config": cfg.to_dict(),
            "error": f"{type(e).__name__}: {e}",
        }
        write_json(out_path, err_payload)
        print(f"  FAILED: {e}")
        results_summary.append(err_payload)

In [ ]:
import pandas as pd
from utils import flatten_runs, load_runs

test_runs = load_runs(RUNS_OUT, status="ok")
df = pd.DataFrame(flatten_runs(test_runs))
cols = ["run_id", "cfg.backend", "cfg.model_precision", "cfg.input_quant_bits",
        "res.top1_acc", "res.top5_acc", "res.infer_ms_avg", "res.infer_ms_std"]
df[[c for c in cols if c in df.columns]].sort_values(
    ["cfg.backend", "cfg.input_quant_bits", "cfg.model_precision"]
).reset_index(drop=True)